In [0]:
gold_df = spark.read.table("realtime_weather.silver.silver_weather_clean")


1.What is the latest weather reading for each city right now?

In [0]:
q1=spark.sql(""" WITH cte AS (
SELECT city_name AS city,country,latitude,longitude,event_timestamp,temperature_celsius,feels_like_celsius,humidity_percent,wind_speed_mps,
pressure_hpa,weather_description,rainfall_mm,weather_severity_score,temperature_band,is_daytime,
ROW_NUMBER() OVER (PARTITION BY city_name ORDER BY event_timestamp DESC) AS rn FROM realtime_weather.silver.silver_weather_clean)

SELECT * FROM cte WHERE rn = 1 """)
q1.display()
q1.write.mode("overwrite").saveAsTable("realtime_weather.gold.gold_city_current")

city_name,country,latitude,longitude,event_timestamp,temperature_celsius,feels_like_celsius,humidity_percent,wind_speed_mps,pressure_hpa,weather_description,rainfall_mm,weather_severity_score,temperature_band,is_daytime,rn
Agra,IN,27.1833,78.0167,2026-04-29T09:29:43.000Z,36.03,37.29,34,2.06,998,scattered clouds,0.0,0,Hot,true,1
Ahmedabad,IN,23.0333,72.6167,2026-04-29T09:29:42.000Z,38.02,36.69,20,4.63,1002,clear sky,0.0,0,Hot,true,1
Aizawl,IN,23.7333,92.7167,2026-04-29T09:29:44.000Z,15.89,15.99,94,4.12,1007,thunderstorm with light rain,1.78,0,Normal,true,1
Akola,IN,20.7333,77.0,2026-04-29T09:29:47.000Z,42.43,39.43,10,7.6,1002,clear sky,0.0,3,Extreme,true,1
Aligarh,IN,27.8833,78.0833,2026-04-29T09:29:48.000Z,38.85,37.08,17,0.55,999,overcast clouds,0.0,0,Hot,true,1
Ambāla,IN,30.3783,76.7808,2026-04-29T09:29:49.000Z,38.31,36.24,16,2.91,999,broken clouds,0.0,0,Hot,true,1
Amritsar,IN,31.6331,74.8656,2026-04-29T09:29:43.000Z,32.97,33.32,38,3.09,1000,few clouds,0.0,0,Hot,true,1
Amrāvati,IN,20.9333,77.75,2026-04-29T09:29:47.000Z,42.09,39.35,11,7.16,1002,clear sky,0.0,3,Extreme,true,1
Anantapur,IN,14.6833,77.6,2026-04-29T09:29:47.000Z,39.43,39.8,24,0.77,1004,clear sky,0.0,0,Hot,true,1
Aurangabad,IN,19.8833,75.3333,2026-04-29T09:29:47.000Z,40.05,38.32,16,0.0,1004,haze,0.0,3,Extreme,true,1


2.What are the hourly averages for each city over the last 24 hours?

In [0]:
q2 = spark.sql("""
SELECT city_name AS city,event_date,event_hour,
    AVG(temperature_celsius) AS avg_temperature,
    AVG(humidity_percent) AS avg_humidity,
    AVG(wind_speed_mps) AS avg_wind_speed,
    AVG(pressure_hpa) AS avg_pressure,
    SUM(rainfall_mm) AS total_rainfall,
    MAX(weather_severity_score) AS max_severity_score
FROM realtime_weather.silver.silver_weather_clean
WHERE event_timestamp >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY city_name,event_date,event_hour """)
q2.display()
q2.write.mode("overwrite").saveAsTable("realtime_weather.gold.gold_hourly_summary")

city,event_date,event_hour,avg_temperature,avg_humidity,avg_wind_speed,avg_pressure,total_rainfall,max_severity_score
Pune,2026-05-10,15,34.483333333333334,24.0,3.3000000000000003,1007.0,0.0,0
Kochi,2026-05-10,16,28.156666666666666,82.33333333333333,1.17,1009.0,0.0,0
Patna,2026-05-10,15,28.626666666666665,66.33333333333333,3.09,1006.3333333333334,0.0,0
Belgaum,2026-05-10,16,27.95,78.83333333333333,5.14,1008.3333333333334,0.0,0
Kadapa,2026-05-10,16,31.228333333333335,54.333333333333336,4.276666666666666,1008.0,0.0,0
Haridwar,2026-05-10,15,26.94,46.5,2.035,1007.0,0.0,0
Shimla,2026-05-10,15,16.560000000000002,53.0,1.28,1011.5,0.0,0
Thoothukudi,2026-05-10,15,29.16,74.0,2.06,1008.0,0.0,0
Kanchipuram,2026-05-10,15,29.21,69.0,5.025,1008.0,0.0,0
Chandigarh,2026-05-10,14,30.79,48.0,2.57,1005.0,0.0,0


3.What were the daily high, low, and peak conditions for each city?

In [0]:
q3 = spark.sql("""
SELECT city_name AS city,event_date,
MAX(temperature_celsius) AS max_temp,
MAX_BY(event_timestamp, temperature_celsius) AS max_temp_time,
MIN(temperature_celsius) AS min_temp,
MAX(wind_speed_mps) AS peak_wind,
MAX_BY(event_timestamp, wind_speed_mps) AS peak_wind_time,
SUM(rainfall_mm) AS total_rainfall,
MAX(weather_severity_score) AS max_severity
FROM realtime_weather.silver.silver_weather_clean
GROUP BY city_name, event_date """)
q3.display()
q3.write.mode("overwrite").saveAsTable("realtime_weather.gold.gold_daily_extremes")

4.Which cities are currently in or recently entered severe weather
conditions?

In [0]:
q4 = spark.sql("""
WITH filtered AS (
    SELECT *FROM realtime_weather.silver.silver_weather_clean
    WHERE event_timestamp >= current_timestamp() - INTERVAL 2 HOURS),
flagged AS (
    SELECT *,
        CASE
            WHEN weather_severity_score >= 4 THEN 'heatwave'
            WHEN LOWER(weather_description) LIKE '%storm%' 
                 OR LOWER(weather_description) LIKE '%thunder%' THEN 'storm'
            WHEN wind_speed_mps > 50 THEN 'high_wind'
            WHEN rainfall_mm > 50 THEN 'heavy_rain'
        END AS alert_type
    FROM filtered WHERE weather_severity_score >= 4
        OR LOWER(weather_description) LIKE '%storm%'
        OR LOWER(weather_description) LIKE '%thunder%'
        OR wind_speed_mps > 50
        OR rainfall_mm > 50),
latest AS (
    SELECT *,ROW_NUMBER() OVER (PARTITION BY city_name ORDER BY event_timestamp DESC) AS rn FROM flagged)
SELECT
    city_name AS city,
    event_timestamp AS event_time,
    temperature_celsius AS temperature,
    wind_speed_mps AS wind_speed,
    rainfall_mm AS rainfall,
    weather_description AS weather_condition,
    weather_severity_score,
    alert_type,
    current_timestamp() AS alert_triggered_at
FROM latest
WHERE rn = 1""")
q4.display()
q4.write.mode("overwrite").saveAsTable("realtime_weather.gold.gold_severe_weather_alerts")

5.Is the API pipeline healthy? Are all cities returning data on schedule?

In [0]:
q5 = spark.sql("""
WITH base AS (
    SELECT city_requested AS city,
        load_date,
        http_status,
        response_latency_ms,
        call_timestamp,
        raw_response
    FROM realtime_weather.bronze.bronze_weather_data),
agg AS (
    SELECT city,load_date,
        COUNT(*) AS total_calls,
        SUM(CASE WHEN http_status = 200 THEN 1 ELSE 0 END) AS successful_calls,
        SUM(CASE WHEN http_status != 200 THEN 1 ELSE 0 END) AS failed_calls,
        SUM(CASE 
            WHEN http_status = 200 AND raw_response IS NULL THEN 1 
            ELSE 0 
        END) AS parse_errors,
        AVG(response_latency_ms) AS avg_latency_ms,
        MAX(CASE 
            WHEN http_status = 200 THEN call_timestamp 
        END) AS last_success_time
    FROM base
    GROUP BY city, load_date)
SELECT
    city,
    load_date,
    total_calls,
    successful_calls,
    failed_calls,
    parse_errors,
    ROUND(avg_latency_ms, 2) AS avg_latency_ms,
    ROUND((successful_calls * 100.0 / total_calls), 2) AS success_rate,
    CASE 
        WHEN last_success_time < current_timestamp() - INTERVAL 1 HOUR 
        THEN true ELSE false 
    END AS is_stale
FROM agg """)
q5.display()
q5.write.mode("overwrite").saveAsTable("realtime_weather.gold.gold_api_health")